# Covid-19 analysis dashboard

## Problem statement

Build a set of linked visualisations that answer: how did case counts evolve over time, which regions were worst affected, and how did the growth rate change after interventions. Data: Johns Hopkins CSSE time series (see `data/README.md`).

### Pipeline

Problem -> Data -> EDA -> Preprocessing -> Modeling -> Evaluation -> Deployment notes -> Conclusion.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(0)
print('matplotlib', matplotlib.__version__)

## Data

Load the time series. A synthetic stand-in is generated here so the notebook runs offline; replace the loader with the real CSV path.

In [ ]:
from pathlib import Path
rng = np.random.default_rng(0)
dates = pd.date_range('2026-01-01', periods=240, freq='D')
regions = ['north', 'south', 'east', 'west']
frames = []
for i, region in enumerate(regions):
    wave = 400 * np.exp(-((np.arange(240) - 60 - i * 25) ** 2) / (2 * 30 ** 2))
    wave += 250 * np.exp(-((np.arange(240) - 170) ** 2) / (2 * 25 ** 2))
    noise = rng.normal(0, 20, 240)
    frames.append(pd.DataFrame({'date': dates, 'region': region,
                                'new_cases': np.clip(wave + noise, 0, None).round()}))
covid = pd.concat(frames, ignore_index=True)
real_path = Path('data/raw/time_series_covid19_confirmed_global.csv')
print('using real data' if real_path.exists() else 'using synthetic stand-in')
print(covid.head())
print('rows:', len(covid))

## EDA

Total trajectory and the shape of each wave.

In [ ]:
totals = covid.groupby('date')['new_cases'].sum()
print(totals.describe().round(1))
peak_date = totals.idxmax()
print('peak day:', peak_date.date(), 'with', int(totals.max()), 'cases')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(totals.index, totals.values, alpha=0.4, label='daily')
ax.plot(totals.index, totals.rolling(7).mean(), linewidth=2, label='7-day mean')
ax.axvline(peak_date, color='crimson', linestyle='--', label='peak')
ax.set_title('Daily new cases')
ax.set_ylabel('cases')
ax.legend()
fig.tight_layout()

## Regional comparison

Small multiples make regional differences legible.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey=True)
for ax, region in zip(axes.flat, covid['region'].unique()):
    sub = covid[covid['region'] == region]
    ax.fill_between(sub['date'], sub['new_cases'].rolling(7).mean(), alpha=0.6)
    ax.set_title(region)
    ax.grid(alpha=0.3)
fig.suptitle('Seven-day average by region')
fig.supylabel('cases')
fig.tight_layout()
print(covid.groupby('region')['new_cases'].sum().astype(int).to_dict())

## Growth rate

Week-over-week growth is the operational metric.

In [ ]:
weekly = covid.groupby([pd.Grouper(key='date', freq='W'), 'region'])['new_cases'].sum()
growth = weekly.groupby('region').pct_change().mul(100).round(1)
table = growth.unstack('region').tail(8)
print(table)
fig, ax = plt.subplots(figsize=(9, 4))
for region in table.columns:
    ax.plot(table.index, table[region], marker='o', label=region)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('week over week change (percent)')
ax.set_title('Growth rate by region')
ax.legend()
fig.tight_layout()

## Heatmap

A calendar heatmap shows intensity and periodicity at once.

In [ ]:
pivot = covid.pivot_table(index='region', columns=covid['date'].dt.isocalendar().week,
                          values='new_cases', aggfunc='sum')
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(pivot, cmap='Reds', ax=ax, cbar_kws={'label': 'weekly cases'})
ax.set_xlabel('ISO week')
ax.set_title('Case intensity by region and week')
fig.tight_layout()
print('weeks covered:', pivot.shape[1])

## Preprocessing

Smooth, index and align the series before modelling.

In [ ]:
wide = covid.pivot_table(index='date', columns='region',
                         values='new_cases', aggfunc='sum').fillna(0)
smoothed = wide.rolling(7, min_periods=1).mean()
indexed = smoothed.div(smoothed.iloc[0].replace(0, np.nan)).fillna(1).round(3)
print(smoothed.tail(3).round(1))
print('any nulls left:', bool(smoothed.isna().any().any()))
print('indexed to day one, final values:', indexed.iloc[-1].round(2).to_dict())

## Simple forecast

A naive seasonal baseline gives the dashboard a forward view.

In [ ]:
history = smoothed.sum(axis=1)
train, test = history.iloc[:-28], history.iloc[-28:]
naive = np.repeat(train.iloc[-7:].mean(), len(test))
drift = train.iloc[-1] + (np.arange(1, len(test) + 1)
                          * (train.iloc[-1] - train.iloc[-28]) / 28)
mae_naive = float(np.mean(np.abs(test.values - naive)))
mae_drift = float(np.mean(np.abs(test.values - drift)))
print(f'naive mean absolute error: {mae_naive:.1f}')
print(f'drift mean absolute error: {mae_drift:.1f}')
print('better baseline:', 'naive' if mae_naive < mae_drift else 'drift')

## Evaluation

Plot the baseline against the held-out window.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-60:], train.values[-60:], label='history')
ax.plot(test.index, test.values, label='actual', linewidth=2)
ax.plot(test.index, naive, linestyle='--', label='naive baseline')
ax.plot(test.index, drift, linestyle=':', label='drift baseline')
ax.set_title('Held-out 28 days against two baselines')
ax.set_ylabel('smoothed daily cases')
ax.legend()
fig.tight_layout()
print('any model shipped to the dashboard must beat', round(min(mae_naive, mae_drift), 1))

## Deployment notes

How this becomes a live dashboard.

In [ ]:
notes = '''
1. Move the loader into src/data_loader.py with a cache so the CSV is fetched once per day.
2. Replace each matplotlib figure with a plotly figure so tooltips and zoom work.
3. Wrap the layout in streamlit: st.plotly_chart(fig, use_container_width=True).
4. Add a sidebar with region multiselect and a date range slider.
5. Schedule a daily refresh; alert if the source file has not changed in 48 hours.
'''
print(notes.strip())
for path in ['app.py', 'src/data_loader.py', 'requirements.txt']:
    print('would create:', path)

## Conclusion

Write three bullets here after you run the notebook: what the model does well, where it fails, and what you would try with one more week.